## 06 — ogbg-molpcba Multi-Task Extension

Extends the GIN/GAT comparison to OGB's `ogbg-molpcba`, a 128-task multi-label bioassay benchmark with ~39% missing labels overall. Reuses the exact GIN and GAT architectures and hyperparameters tuned on `molhiv` (notebooks 02-03), swapping only the output layer to 128 logits and the loss to a masked, per-task-averaged `BCEWithLogitsLoss`. Reports mean Average Precision (via OGB's `Evaluator`) on the official scaffold-split test set for both models, alongside a per-task AP spread analysis and an honest comparison against the public leaderboard.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = "/content/drive/MyDrive/molhiv-gnn"

# !pip install torch-geometric ogb rdkit

BASE_DIR = ".."

In [2]:
import pandas as pd
import os
import json
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from sklearn.metrics import average_precision_score
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GATConv, global_mean_pool
from ogb.graphproppred import PygGraphPropPredDataset, Evaluator
from ogb.graphproppred.mol_encoder import AtomEncoder

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
device

device(type='cuda')

In [3]:
dataset = PygGraphPropPredDataset(name="ogbg-molpcba", root=f"{BASE_DIR}/dataset")
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name="ogbg-molpcba")
num_tasks = dataset.num_tasks

train_loader = DataLoader(dataset[split_idx["train"]], batch_size=512, shuffle=True)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=512, shuffle=False)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=512, shuffle=False)

len(dataset), {k: len(v) for k, v in split_idx.items()}, num_tasks

(437929, {'train': 350343, 'valid': 43793, 'test': 43793}, 128)

In [4]:
batch = next(iter(train_loader))
print("batch.y shape:", batch.y.shape)

y_all = dataset._data.y.float()
nan_mask = torch.isnan(y_all)
print("overall NaN fraction across all molecules/tasks:", nan_mask.float().mean().item())

batch.y shape: torch.Size([512, 128])
overall NaN fraction across all molecules/tasks: 0.3931455612182617


In [5]:
valid_per_task = (~nan_mask).sum(dim=0)
print("valid labels per task -> min: {}  max: {}  mean: {:.1f}".format(
    valid_per_task.min().item(), valid_per_task.max().item(), valid_per_task.float().mean().item()
))
print("ratio of most-labeled task to least-labeled task: {:.1f}x".format(
    valid_per_task.max().item() / valid_per_task.min().item()
))

valid labels per task -> min: 7563  max: 411867  mean: 265759.1
ratio of most-labeled task to least-labeled task: 54.5x


In [6]:
pos_count = ((y_all == 1) & (~nan_mask)).sum(dim=0).float()
pos_rate_per_task = pos_count / valid_per_task.float().clamp(min=1)
print("positive rate among valid labels, per task -> min: {:.6f}  max: {:.4f}  mean: {:.4f}".format(
    pos_rate_per_task.min().item(), pos_rate_per_task.max().item(), pos_rate_per_task.mean().item()
))

positive rate among valid labels, per task -> min: 0.000094  max: 0.3312  mean: 0.0216


In [7]:
def masked_multitask_loss(pred, target):
    mask = ~torch.isnan(target)
    safe_target = torch.nan_to_num(target, nan=0.0)
    elementwise = F.binary_cross_entropy_with_logits(pred, safe_target, reduction="none")
    elementwise = elementwise * mask.float()
    valid_per_task = mask.sum(dim=0)
    task_has_valid = valid_per_task > 0
    per_task_loss = elementwise.sum(dim=0) / valid_per_task.clamp(min=1)
    return per_task_loss[task_has_valid].mean()

In [8]:
class GIN(nn.Module):
    def __init__(self, hidden_dim, num_layers, dropout, out_dim):
        super().__init__()
        self.atom_encoder = AtomEncoder(hidden_dim)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.dropout = dropout
        self.out = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        h = self.atom_encoder(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index)
            h = bn(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        h = global_mean_pool(h, batch)
        return self.out(h)


class GAT(nn.Module):
    def __init__(self, hidden_dim, num_layers, dropout, heads, out_dim):
        super().__init__()
        self.atom_encoder = AtomEncoder(hidden_dim)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        in_dim = hidden_dim
        for i in range(num_layers):
            is_last = i == num_layers - 1
            concat = not is_last
            conv = GATConv(in_dim, hidden_dim, heads=heads, concat=concat, dropout=dropout)
            self.convs.append(conv)
            in_dim = hidden_dim * heads if concat else hidden_dim
            self.bns.append(nn.BatchNorm1d(in_dim))
        self.dropout = dropout
        self.out = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, batch):
        h = self.atom_encoder(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index)
            h = bn(h)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        h = global_mean_pool(h, batch)
        return self.out(h)

In [9]:
_sanity_model = GIN(hidden_dim=256, num_layers=5, dropout=0.2, out_dim=num_tasks).to(device)
_sanity_opt = torch.optim.Adam(_sanity_model.parameters(), lr=1e-3)
_sanity_model.train()

for i, sanity_batch in enumerate(train_loader):
    sanity_batch = sanity_batch.to(device)
    _sanity_opt.zero_grad()
    sanity_out = _sanity_model(sanity_batch.x, sanity_batch.edge_index, sanity_batch.batch)
    sanity_loss = masked_multitask_loss(sanity_out, sanity_batch.y.float())
    assert torch.isfinite(sanity_loss), f"non-finite loss at step {i}: {sanity_loss}"
    sanity_loss.backward()
    _sanity_opt.step()
    print(f"sanity step {i}: loss = {sanity_loss.item():.4f}, finite = {torch.isfinite(sanity_loss).item()}")
    if i >= 4:
        break

print("all sanity steps finite, proceeding to full training")
del _sanity_model, _sanity_opt

sanity step 0: loss = 0.7074, finite = True


sanity step 1: loss = 0.6614, finite = True


sanity step 2: loss = 0.6165, finite = True
sanity step 3: loss = 0.5715, finite = True


sanity step 4: loss = 0.5297, finite = True
all sanity steps finite, proceeding to full training


In [10]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = masked_multitask_loss(out, batch.y.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        y_true.append(batch.y.cpu())
        y_pred.append(torch.sigmoid(out).cpu())
    return torch.cat(y_true, dim=0).numpy(), torch.cat(y_pred, dim=0).numpy()


def evaluate(model, loader):
    y_true, y_pred = get_predictions(model, loader)
    return evaluator.eval({"y_true": y_true, "y_pred": y_pred})["ap"]


def per_task_ap(y_true, y_pred):
    ap_list = []
    task_indices = []
    for t in range(y_true.shape[1]):
        col_true = y_true[:, t]
        is_labeled = ~np.isnan(col_true)
        col_true = col_true[is_labeled]
        col_pred = y_pred[is_labeled, t]
        if col_true.sum() > 0 and (col_true == 0).sum() > 0:
            ap_list.append(average_precision_score(col_true, col_pred))
            task_indices.append(t)
    return np.array(ap_list), np.array(task_indices)

In [11]:
MAX_EPOCHS = 20
PATIENCE = 5

gin_model = GIN(hidden_dim=256, num_layers=5, dropout=0.2, out_dim=num_tasks).to(device)
gin_optimizer = torch.optim.Adam(gin_model.parameters(), lr=1e-3)

gin_best_val_ap = 0.0
gin_best_state = None
patience_ctr = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_epoch(gin_model, train_loader, gin_optimizer)
    val_ap = evaluate(gin_model, valid_loader)

    if val_ap > gin_best_val_ap:
        gin_best_val_ap = val_ap
        gin_best_state = copy.deepcopy(gin_model.state_dict())
        patience_ctr = 0
    else:
        patience_ctr += 1

    print(f"[GIN] epoch={epoch:>3}  train_loss={train_loss:.4f}  val_ap={val_ap:.4f}  best_val_ap={gin_best_val_ap:.4f}")

    if patience_ctr >= PATIENCE:
        print(f"early stopping at epoch {epoch}")
        break

print(f"GIN finished with best val AP = {gin_best_val_ap:.4f}")

[GIN] epoch=  1  train_loss=0.0795  val_ap=0.0537  best_val_ap=0.0537


[GIN] epoch=  2  train_loss=0.0650  val_ap=0.0717  best_val_ap=0.0717


[GIN] epoch=  3  train_loss=0.0630  val_ap=0.0825  best_val_ap=0.0825


[GIN] epoch=  4  train_loss=0.0617  val_ap=0.0874  best_val_ap=0.0874


[GIN] epoch=  5  train_loss=0.0608  val_ap=0.1023  best_val_ap=0.1023


...

[GIN] epoch= 16  train_loss=0.0547  val_ap=0.1371  best_val_ap=0.1371


[GIN] epoch= 17  train_loss=0.0545  val_ap=0.1311  best_val_ap=0.1371


[GIN] epoch= 18  train_loss=0.0542  val_ap=0.1372  best_val_ap=0.1372


[GIN] epoch= 19  train_loss=0.0538  val_ap=0.1388  best_val_ap=0.1388


[GIN] epoch= 20  train_loss=0.0535  val_ap=0.1449  best_val_ap=0.1449
GIN finished with best val AP = 0.1449


In [12]:
gin_model.load_state_dict(gin_best_state)
gin_y_true, gin_y_pred = get_predictions(gin_model, test_loader)
gin_test_ap = evaluator.eval({"y_true": gin_y_true, "y_pred": gin_y_pred})["ap"]
gin_per_task_ap, gin_task_indices = per_task_ap(gin_y_true, gin_y_pred)

print("GIN test mean AP:", gin_test_ap)
print("GIN per-task AP -> min: {:.4f}  max: {:.4f}  median: {:.4f}  n_tasks_scored: {}".format(
    gin_per_task_ap.min(), gin_per_task_ap.max(), np.median(gin_per_task_ap), len(gin_per_task_ap)
))

GIN test mean AP: 0.14918654197643494
GIN per-task AP -> min: 0.0002  max: 0.7825  median: 0.1026  n_tasks_scored: 127


In [13]:
gat_model = GAT(hidden_dim=64, num_layers=3, dropout=0.2, heads=4, out_dim=num_tasks).to(device)
gat_optimizer = torch.optim.Adam(gat_model.parameters(), lr=1e-3)

gat_best_val_ap = 0.0
gat_best_state = None
patience_ctr = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_epoch(gat_model, train_loader, gat_optimizer)
    val_ap = evaluate(gat_model, valid_loader)

    if val_ap > gat_best_val_ap:
        gat_best_val_ap = val_ap
        gat_best_state = copy.deepcopy(gat_model.state_dict())
        patience_ctr = 0
    else:
        patience_ctr += 1

    print(f"[GAT] epoch={epoch:>3}  train_loss={train_loss:.4f}  val_ap={val_ap:.4f}  best_val_ap={gat_best_val_ap:.4f}")

    if patience_ctr >= PATIENCE:
        print(f"early stopping at epoch {epoch}")
        break

print(f"GAT finished with best val AP = {gat_best_val_ap:.4f}")

[GAT] epoch=  1  train_loss=0.1277  val_ap=0.0632  best_val_ap=0.0632


[GAT] epoch=  2  train_loss=0.0667  val_ap=0.0695  best_val_ap=0.0695


[GAT] epoch=  3  train_loss=0.0645  val_ap=0.0794  best_val_ap=0.0794


[GAT] epoch=  4  train_loss=0.0633  val_ap=0.0856  best_val_ap=0.0856


[GAT] epoch=  5  train_loss=0.0627  val_ap=0.0920  best_val_ap=0.0920


...

[GAT] epoch= 16  train_loss=0.0599  val_ap=0.1089  best_val_ap=0.1106


[GAT] epoch= 17  train_loss=0.0597  val_ap=0.1120  best_val_ap=0.1120


[GAT] epoch= 18  train_loss=0.0597  val_ap=0.1135  best_val_ap=0.1135


[GAT] epoch= 19  train_loss=0.0595  val_ap=0.1140  best_val_ap=0.1140


[GAT] epoch= 20  train_loss=0.0592  val_ap=0.1128  best_val_ap=0.1140
GAT finished with best val AP = 0.1140


In [14]:
gat_model.load_state_dict(gat_best_state)
gat_y_true, gat_y_pred = get_predictions(gat_model, test_loader)
gat_test_ap = evaluator.eval({"y_true": gat_y_true, "y_pred": gat_y_pred})["ap"]
gat_per_task_ap, gat_task_indices = per_task_ap(gat_y_true, gat_y_pred)

print("GAT test mean AP:", gat_test_ap)
print("GAT per-task AP -> min: {:.4f}  max: {:.4f}  median: {:.4f}  n_tasks_scored: {}".format(
    gat_per_task_ap.min(), gat_per_task_ap.max(), np.median(gat_per_task_ap), len(gat_per_task_ap)
))

GAT test mean AP: 0.11789729706992944
GAT per-task AP -> min: 0.0003  max: 0.7132  median: 0.0687  n_tasks_scored: 127


In [15]:
print(f"{'Model':<6}{'Test mean AP':>14}")
print(f"{'GIN':<6}{gin_test_ap:>14.4f}")
print(f"{'GAT':<6}{gat_test_ap:>14.4f}")

Model   Test mean AP
GIN           0.1492
GAT           0.1179


In [16]:
gin_task_valid_counts = valid_per_task[gin_task_indices].numpy()
gin_corr = np.corrcoef(np.log10(gin_task_valid_counts), gin_per_task_ap)[0, 1]

gat_task_valid_counts = valid_per_task[gat_task_indices].numpy()
gat_corr = np.corrcoef(np.log10(gat_task_valid_counts), gat_per_task_ap)[0, 1]

print("correlation between log10(valid label count) and per-task AP:")
print(f"  GIN: {gin_corr:.3f}")
print(f"  GAT: {gat_corr:.3f}")

correlation between log10(valid label count) and per-task AP:
  GIN: -0.460
  GAT: -0.477


In [17]:
leaderboard_snapshot = [
    ("HyperFusion (rank 1)", 0.3204, 0.0001),
    ("HIG, pretrained on PCQM4M (rank 2)", 0.3167, 0.0034),
    ("Graphormer, pretrained on PCQM4M (rank 3)", 0.3140, 0.0032),
    ("GIN+virtual node (OGB team baseline)", 0.2703, 0.0023),
    ("GCN+virtual node (OGB team baseline)", 0.2424, 0.0034),
    ("GIN (OGB team baseline, no virtual node)", 0.2266, 0.0028),
    ("GCN (OGB team baseline, no virtual node)", 0.2020, 0.0024),
    ("This project — GIN", gin_test_ap, None),
    ("This project — GAT", gat_test_ap, None),
]

print(f"{'Entry':<45}{'Test mean AP':>14}")
for name, ap, std in leaderboard_snapshot:
    ap_str = f"{ap:.4f} ± {std:.4f}" if std is not None else f"{ap:.4f}"
    print(f"{name:<45}{ap_str:>18}")

Entry                                          Test mean AP
HyperFusion (rank 1)                            0.3204 ± 0.0001
HIG, pretrained on PCQM4M (rank 2)              0.3167 ± 0.0034
Graphormer, pretrained on PCQM4M (rank 3)       0.3140 ± 0.0032
GIN+virtual node (OGB team baseline)            0.2703 ± 0.0023
GCN+virtual node (OGB team baseline)            0.2424 ± 0.0034
GIN (OGB team baseline, no virtual node)        0.2266 ± 0.0028
GCN (OGB team baseline, no virtual node)        0.2020 ± 0.0024
This project — GIN                                       0.1492
This project — GAT                                       0.1179


In [18]:
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)

torch.save(gin_best_state, f"{BASE_DIR}/models/gin_molpcba_best.pt")
torch.save(gat_best_state, f"{BASE_DIR}/models/gat_molpcba_best.pt")

results_molpcba = {
    "gin": {
        "config": {"hidden_dim": 256, "num_layers": 5, "dropout": 0.2},
        "val_ap": gin_best_val_ap,
        "test_ap": gin_test_ap,
        "per_task_ap": gin_per_task_ap.tolist(),
        "per_task_ap_task_indices": gin_task_indices.tolist(),
    },
    "gat": {
        "config": {"hidden_dim": 64, "num_layers": 3, "dropout": 0.2, "heads": 4},
        "val_ap": gat_best_val_ap,
        "test_ap": gat_test_ap,
        "per_task_ap": gat_per_task_ap.tolist(),
        "per_task_ap_task_indices": gat_task_indices.tolist(),
    },
}

with open(f"{BASE_DIR}/outputs/results_molpcba.json", "w") as f:
    json.dump(results_molpcba, f, indent=2)

{"gin_test_ap": gin_test_ap, "gat_test_ap": gat_test_ap}

{'gin_test_ap': 0.14918654197643494, 'gat_test_ap': 0.11789729706992944}